# UROP MATR Anomaly Detection - Local Version

이 노트북은 로컬 폴더에 있는 프로젝트 코드와 MATR 데이터셋을 바로 사용한다.

기본 경로:

```text
PROJECT_DIR = 현재 작업 폴더, 또는 C:/Users/kyucho/UROP 자동 탐색
MATR_DIR    = PROJECT_DIR/MATR
```

다른 위치를 쓰려면 첫 번째 코드 셀을 실행하기 전에 환경변수 `PROJECT_DIR`, `MATR_DIR`를 지정하면 된다.


## 1. 로컬 경로 설정


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

def resolve_project_dir():
    candidates = []

    env_project = os.environ.get('PROJECT_DIR')
    if env_project:
        candidates.append(Path(env_project))

    candidates.extend([
        Path(r'C:/Users/kyucho/UROP'),
        Path.home() / 'UROP',
        Path.cwd(),
    ])

    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'scripts' / 'run_matr_locked_test_evaluation.py').exists():
            return candidate

    checked = '\n'.join(str(p.expanduser()) for p in candidates)
    raise FileNotFoundError(
        'Could not find project directory containing '
        'scripts/run_matr_locked_test_evaluation.py. Checked:\n'
        + checked
    )

PROJECT_DIR = resolve_project_dir()
MATR_DIR = Path(os.environ.get('MATR_DIR', PROJECT_DIR / 'MATR')).expanduser().resolve()

print('PROJECT_DIR:', PROJECT_DIR)
print('MATR_DIR:', MATR_DIR)
print('PROJECT_DIR exists:', PROJECT_DIR.exists())
print('MATR_DIR exists:', MATR_DIR.exists())


## 2. 로컬 파일 및 설정 확인


In [ ]:
# Current SOH configuration:
# - battery-level split
# - sliding-window samples
# - lookback=10
# - horizons=50,100
# - target_scale=100, fixed_len=100
CONFIG_DIR = PROJECT_DIR / 'outputs' / 'matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide'
CONFIG_PATH = CONFIG_DIR / 'best_optuna_config.json'
TUNING_CONTEXT_PATH = CONFIG_DIR / 'optuna_tuning_config.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'matr_locked_test_sliding_l10_h50_h100_wide_best_anomaly'
RUN_SCRIPT = PROJECT_DIR / 'scripts' / 'run_matr_locked_test_evaluation.py'

# MATR data is expected to be an already-extracted local folder.
pkl_files = sorted(MATR_DIR.rglob('*.pkl')) if MATR_DIR.exists() else []

print('PROJECT_DIR:', PROJECT_DIR)
print('MATR_DIR:', MATR_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RUN_SCRIPT exists:', RUN_SCRIPT.exists(), RUN_SCRIPT)
print('CONFIG_PATH exists:', CONFIG_PATH.exists(), CONFIG_PATH)
print('TUNING_CONTEXT_PATH exists:', TUNING_CONTEXT_PATH.exists(), TUNING_CONTEXT_PATH)
print('PKL file count:', len(pkl_files))
for path in pkl_files[:10]:
    print(path)

if not RUN_SCRIPT.exists():
    raise FileNotFoundError(f'Missing run script: {RUN_SCRIPT}')
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        'Missing tuned config file. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/best_optuna_config.json '
        'inside the uploaded project folder.'
    )
if not TUNING_CONTEXT_PATH.exists():
    raise FileNotFoundError(
        'Missing Optuna sidecar config. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/optuna_tuning_config.json '
        'next to best_optuna_config.json.'
    )
if not pkl_files:
    raise RuntimeError(
        f'No .pkl files found under MATR_DIR={MATR_DIR}. '
        'Place the already-extracted MATR dataset folder there, or set os.environ["MATR_DIR"] before running setup.'
    )


## 3. GPU 확인


In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 4. 최신 코드 실행


In [ ]:
help_result = subprocess.run(
    [sys.executable, str(RUN_SCRIPT), '--help'],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)

help_text = help_result.stdout + '\n' + help_result.stderr
required_options = ['--sample-mode', '--lookback', '--horizons', '--include-references']
missing_options = [opt for opt in required_options if opt not in help_text]
if missing_options:
    raise RuntimeError(
        'The evaluation script is not the updated version. Missing options: '
        + ', '.join(missing_options)
        + '\n\nSTDERR:\n'
        + help_result.stderr[-4000:]
    )

config_rel = CONFIG_PATH.relative_to(PROJECT_DIR).as_posix()
output_rel = OUTPUT_DIR.relative_to(PROJECT_DIR).as_posix()

cmd = [
    sys.executable,
    str(RUN_SCRIPT),
    '--data-root', str(MATR_DIR),
    '--config-path', config_rel,
    '--output-dir', output_rel,
    '--device', DEVICE,
    '--include-references',
    '--sample-mode', 'sliding-window',
    '--lookback', '10',
    '--horizons', '50', '100',
    '--seeds', '42', '43', '44',
    '--target-scale', '100',
    '--fixed-len', '100',
]

# If the script has native anomaly flags, pass the same threshold settings.
# Otherwise, the notebook computes residual-based anomaly scores below.
if all(opt in help_text for opt in ['--anomaly-threshold-method', '--anomaly-score-mode', '--cell-anomaly-ratio-threshold']):
    cmd += [
        '--anomaly-threshold-method', 'top5',
        '--anomaly-score-mode', 'degradation',
        '--cell-anomaly-ratio-threshold', '0.30',
    ]

print('Running command:')
print(' '.join(cmd))

result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('return code:', result.returncode)
print('\n--- STDOUT tail ---')
print(result.stdout[-8000:])
print('\n--- STDERR tail ---')
print(result.stderr[-8000:])

if result.returncode != 0:
    raise RuntimeError('Model evaluation command failed. Check STDERR above.')


## 5. 필수 결과 파일 확인


In [ ]:
import json

required_outputs = [
    OUTPUT_DIR / 'locked_test_summary.csv',
    OUTPUT_DIR / 'test_predictions.csv',
    OUTPUT_DIR / 'test_summary_by_model_horizon.csv',
    OUTPUT_DIR / 'locked_test_config.json',
]

for path in required_outputs:
    print(path.name, path.exists(), path)

missing = [path for path in required_outputs if not path.exists()]
if missing:
    print('OUTPUT_DIR listing:')
    if OUTPUT_DIR.exists():
        for p in sorted(OUTPUT_DIR.rglob('*'))[:200]:
            print(p.relative_to(OUTPUT_DIR), 'dir' if p.is_dir() else p.stat().st_size)
    raise FileNotFoundError('Missing result files: ' + ', '.join(str(p) for p in missing))

with open(OUTPUT_DIR / 'locked_test_config.json', 'r', encoding='utf-8') as f:
    locked_test_config = json.load(f)

runtime_config = locked_test_config.get('runtime_config', {})
print('Runtime config:', runtime_config)

expected_runtime = {
    'lookback': 10,
    'sample_mode': 'sliding-window',
    'horizons': [50, 100],
    'seeds': [42, 43, 44],
    'target_scale': 100.0,
    'fixed_len': 100,
}

for key, expected in expected_runtime.items():
    actual = runtime_config.get(key)
    if actual != expected:
        raise RuntimeError(f'Unexpected runtime_config[{key!r}]: expected {expected!r}, got {actual!r}')

print('Required model result files exist and runtime config matches the intended sliding-window l10 h50/h100 setup.')


## 6. 기존 모델 vs 개선 모델 성능 비교


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary_df = pd.read_csv(OUTPUT_DIR / 'locked_test_summary.csv')
horizon_df = pd.read_csv(OUTPUT_DIR / 'test_summary_by_model_horizon.csv')
pred = pd.read_csv(OUTPUT_DIR / 'test_predictions.csv')

model_order = ['persistence', 'log_degradation', 'cpmlp', 'cpmlp_cpdsconv_fusion']
model_labels = {
    'persistence': 'Persistence',
    'log_degradation': 'Log Degradation',
    'cpmlp': 'CPMLP',
    'cpmlp_cpdsconv_fusion': 'CPMLP-CPDSConv Fusion',
}
model_colors = {
    'persistence': '#9CA3AF',
    'log_degradation': '#F59E0B',
    'cpmlp': '#10B981',
    'cpmlp_cpdsconv_fusion': '#2563EB',
}

available_order = [m for m in model_order if m in set(summary_df['model'])]
perf = summary_df[summary_df['model'].isin(available_order)].copy()
perf = perf.set_index('model').loc[available_order].reset_index()

display_cols = [c for c in ['model', 'avg_MAE_mean', 'avg_RMSE_mean', 'avg_MAPE_percent_mean', 'average_Skill_MAE_vs_persistence'] if c in perf.columns]
display(perf[display_cols])

if {'cpmlp', 'cpmlp_cpdsconv_fusion'}.issubset(set(summary_df['model'])):
    cpmlp_row = summary_df[summary_df['model'] == 'cpmlp'].iloc[0]
    fusion_row = summary_df[summary_df['model'] == 'cpmlp_cpdsconv_fusion'].iloc[0]
    compare_rows = []
    for col, label in [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE')]:
        if col in summary_df.columns:
            compare_rows.append({
                'metric': label,
                'cpmlp': cpmlp_row[col],
                'cpmlp_cpdsconv_fusion': fusion_row[col],
                'delta_percent_vs_cpmlp': (cpmlp_row[col] - fusion_row[col]) / cpmlp_row[col] * 100,
            })
    comparison_df = pd.DataFrame(compare_rows)
    display(comparison_df)
    comparison_df.to_csv(REPORT_DIR / 'fusion_vs_cpmlp_metrics.csv', index=False, encoding='utf-8-sig')

metric_cols = [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE (%)')]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (col, label) in zip(axes, metric_cols):
    if col not in perf.columns:
        ax.set_visible(False)
        continue
    bars = ax.bar(
        perf['model'].map(model_labels),
        perf[col],
        color=[model_colors[m] for m in perf['model']],
    )
    ax.set_title(label)
    ax.set_ylabel(label)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=18)
    for bar in bars:
        h = bar.get_height()
        txt = f'{h:.5f}' if label != 'MAPE (%)' else f'{h:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha='center', va='bottom', fontsize=9)
fig.suptitle('Model Performance Comparison: Baselines vs Fusion Model', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_model_performance_comparison.png', dpi=220)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
hmetric_cols = [('MAE_mean', 'MAE'), ('RMSE_mean', 'RMSE'), ('MAPE_percent_mean', 'MAPE (%)')]
for ax, (col, label) in zip(axes, hmetric_cols):
    if col not in horizon_df.columns:
        ax.set_visible(False)
        continue
    for model in available_order:
        sub = horizon_df[horizon_df['model'] == model].sort_values('horizon')
        if sub.empty:
            continue
        ax.plot(sub['horizon'], sub[col], marker='o', linewidth=2, label=model_labels[model], color=model_colors[model])
    ax.set_title(label)
    ax.set_xlabel('Prediction Horizon')
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Performance by Horizon', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '02_performance_by_horizon.png', dpi=220)
plt.show()


## 7. Residual 기반 이상탐지 파이프라인


In [ ]:
target_model = 'cpmlp_cpdsconv_fusion'
df = pred[pred['model'] == target_model].copy()

if df.empty:
    raise RuntimeError(f'No predictions found for {target_model}')

if 'cell_id' not in df.columns:
    df['cell_id'] = df['battery_id'].astype(str)

if 'sample_mode' in df.columns:
    sample_modes = sorted(df['sample_mode'].dropna().astype(str).unique())
    print('Prediction sample modes:', sample_modes)
    if sample_modes != ['sliding-window']:
        raise RuntimeError(f'Expected sliding-window predictions, got sample_mode={sample_modes}')
else:
    print('Warning: sample_mode column is missing from predictions. Check locked_test_config.json above.')

if 'horizon' in df.columns:
    observed_horizons = sorted(df['horizon'].dropna().astype(int).unique())
    print('Prediction horizons:', observed_horizons)
    if observed_horizons != [50, 100]:
        raise RuntimeError(f'Expected horizons [50, 100], got {observed_horizons}')

sort_cols = [c for c in ['seed', 'model', 'battery_id', 'horizon', 'target_cycle'] if c in df.columns]
df = df.sort_values(sort_cols).copy()

# Residual-based anomaly scoring:
# this flags prediction-error outliers, not ground-truth physical battery defects.
if 'actual_delta_soh' not in df.columns or 'pred_delta_soh' not in df.columns:
    group_cols = [c for c in ['seed', 'model', 'battery_id', 'horizon'] if c in df.columns]
    df['actual_delta_soh'] = (-df.groupby(group_cols)['actual_soh'].diff()).fillna(0)
    df['pred_delta_soh'] = (-df.groupby(group_cols)['pred_soh'].diff()).fillna(0)

group_cols = [c for c in ['seed', 'model', 'battery_id', 'horizon'] if c in df.columns]
df['residual_score'] = (df['actual_delta_soh'] - df['pred_delta_soh']).abs()
df['degradation_residual'] = (df['actual_delta_soh'] - df['pred_delta_soh']).clip(lower=0.0)
df['actual_soh_step'] = df.groupby(group_cols)['actual_soh'].diff().fillna(0.0)
df['pred_soh_step'] = df.groupby(group_cols)['pred_soh'].diff().fillna(0.0)
df['actual_degradation_step'] = (-df['actual_soh_step']).clip(lower=0.0)
df['pred_degradation_step'] = (-df['pred_soh_step']).clip(lower=0.0)
df['degradation_slope_score'] = (df['actual_degradation_step'] - df['pred_degradation_step']).clip(lower=0.0)

def z_by_seed_horizon(frame, col):
    keys = [c for c in ['seed', 'horizon'] if c in frame.columns]
    if not keys:
        std = frame[col].std(ddof=0)
        return (frame[col] - frame[col].mean()) / (std if std and not np.isnan(std) else 1)
    def transform(s):
        std = s.std(ddof=0)
        if std == 0 or np.isnan(std):
            return s * 0
        return (s - s.mean()) / std
    return frame.groupby(keys)[col].transform(transform)

df['degradation_residual_z'] = z_by_seed_horizon(df, 'degradation_residual').clip(lower=0)
df['degradation_slope_z'] = z_by_seed_horizon(df, 'degradation_slope_score').clip(lower=0)

ALPHA = 0.85
BETA = 0.15
df['degradation_anomaly_score'] = ALPHA * df['degradation_residual_z'] + BETA * df['degradation_slope_z']

threshold_keys = [c for c in ['seed', 'horizon'] if c in df.columns]
if threshold_keys:
    df['threshold_top5'] = df.groupby(threshold_keys)['degradation_anomaly_score'].transform(lambda s: s.quantile(0.95))
else:
    df['threshold_top5'] = df['degradation_anomaly_score'].quantile(0.95)

df['is_cycle_anomaly_top5'] = df['degradation_anomaly_score'] >= df['threshold_top5']

cell_summary = (
    df.groupby(['seed', 'battery_id', 'cell_id'], as_index=False)
    .agg(
        n_points=('target_cycle', 'count'),
        n_anomalies=('is_cycle_anomaly_top5', 'sum'),
        mean_anomaly_score=('degradation_anomaly_score', 'mean'),
        max_anomaly_score=('degradation_anomaly_score', 'max'),
        mean_degradation_residual=('degradation_residual', 'mean'),
        max_degradation_residual=('degradation_residual', 'max'),
        mean_residual_score=('residual_score', 'mean'),
        max_residual_score=('residual_score', 'max'),
    )
)
cell_summary['anomaly_ratio'] = cell_summary['n_anomalies'] / cell_summary['n_points']
cell_summary['is_cell_anomaly'] = cell_summary['anomaly_ratio'] >= 0.30
cell_summary = cell_summary.sort_values(['is_cell_anomaly', 'anomaly_ratio', 'max_anomaly_score'], ascending=[False, False, False])

df.to_csv(REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv', index=False, encoding='utf-8-sig')
cell_summary.to_csv(REPORT_DIR / 'cell_level_anomaly_summary.csv', index=False, encoding='utf-8-sig')

print('Cycle anomaly count:', int(df['is_cycle_anomaly_top5'].sum()), '/', len(df))
print('Cell anomaly count:', int(cell_summary['is_cell_anomaly'].sum()), '/', len(cell_summary))
print('Saved:', REPORT_DIR)

display(cell_summary.head(30))


## 8. 이상탐지 후보 선정 결과 그래프


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# Load analysis results
# =========================
REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

cycle_path = REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv'
cell_path = REPORT_DIR / 'cell_level_anomaly_summary.csv'

if 'df' not in globals():
    df = pd.read_csv(cycle_path)

if 'cell_summary' not in globals():
    cell_summary = pd.read_csv(cell_path)

# =========================
# Candidate selection
# =========================
cand = cell_summary[cell_summary['is_cell_anomaly'] == True].copy()

# 후보가 너무 적으면 anomaly_ratio 기준 상위 cell도 같이 표시
if cand.empty:
    cand = cell_summary.sort_values(
        ['anomaly_ratio', 'max_anomaly_score'],
        ascending=[False, False]
    ).head(12).copy()
else:
    cand = cand.sort_values(
        ['anomaly_ratio', 'max_anomaly_score'],
        ascending=[False, False]
    ).head(12).copy()

# x축 label
cand['label'] = cand['cell_id'].astype(str) + '\nseed=' + cand['seed'].astype(str)

# n_anomaly_horizons 계산
# Count how many evaluated horizons contain at least one anomalous window.
horizon_count = (
    df[df['is_cycle_anomaly_top5'] == True]
    .groupby(['seed', 'battery_id', 'cell_id'], as_index=False)
    .agg(n_anomaly_horizons=('horizon', 'nunique'))
)

total_horizon_count = (
    df.groupby(['seed', 'battery_id', 'cell_id'], as_index=False)
    .agg(n_total_horizons=('horizon', 'nunique'))
)

cand = cand.merge(
    horizon_count,
    on=['seed', 'battery_id', 'cell_id'],
    how='left'
)

cand = cand.merge(
    total_horizon_count,
    on=['seed', 'battery_id', 'cell_id'],
    how='left'
)

cand['n_anomaly_horizons'] = cand['n_anomaly_horizons'].fillna(0).astype(int)
cand['n_total_horizons'] = cand['n_total_horizons'].fillna(0).astype(int)

# 보기 좋은 순서: anomaly ratio 높은 순
cand = cand.sort_values(
    ['anomaly_ratio', 'n_anomaly_horizons', 'max_anomaly_score'],
    ascending=[False, False, False]
).reset_index(drop=True)

display(cand)

# =========================
# Plot
# =========================
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))

# 1. anomaly ratio
axes[0].bar(
    cand['label'],
    cand['anomaly_ratio'],
    color='#2563EB'
)
axes[0].axhline(
    0.30,
    color='red',
    linestyle='--',
    linewidth=1.5,
    label='ratio threshold=0.30'
)
axes[0].set_title('Cell-level Anomaly Ratio')
axes[0].set_ylabel('Anomaly Ratio')
axes[0].set_ylim(0, max(1.05, cand['anomaly_ratio'].max() + 0.1))
axes[0].tick_params(axis='x', rotation=50)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

for i, v in enumerate(cand['anomaly_ratio']):
    axes[0].text(i, v + 0.025, f'{v:.2f}', ha='center', fontsize=9)

# 2. max anomaly score
axes[1].bar(
    cand['label'],
    cand['max_anomaly_score'],
    color='#F97316'
)
axes[1].set_title('Max Anomaly Score')
axes[1].set_ylabel('Max Score')
axes[1].tick_params(axis='x', rotation=50)
axes[1].grid(axis='y', alpha=0.3)

score_margin = max(cand['max_anomaly_score'].max() * 0.08, 0.1)
axes[1].set_ylim(0, cand['max_anomaly_score'].max() + score_margin)

for i, v in enumerate(cand['max_anomaly_score']):
    axes[1].text(i, v + score_margin * 0.15, f'{v:.2f}', ha='center', fontsize=9)

# 3. anomalous horizons
axes[2].bar(
    cand['label'],
    cand['n_anomaly_horizons'],
    color='#16A34A'
)
axes[2].set_title('Number of Anomalous Horizons')
axes[2].set_ylabel('Anomaly Horizons')
axes[2].tick_params(axis='x', rotation=50)
axes[2].grid(axis='y', alpha=0.3)

max_total_h = int(max(cand['n_total_horizons'].max(), 1))
axes[2].set_ylim(0, max_total_h + 0.5)
axes[2].set_yticks(range(0, max_total_h + 1))

for i, row in cand.iterrows():
    axes[2].text(
        i,
        row['n_anomaly_horizons'] + 0.08,
        f"{int(row['n_anomaly_horizons'])}/{int(row['n_total_horizons'])}",
        ha='center',
        fontsize=9
    )

fig.suptitle('Cell-level Anomaly Candidates with Alpha=0.85, Beta=0.15', fontsize=16)
plt.tight_layout()

save_path = FIGURE_DIR / 'cell_level_anomaly_candidates_alpha085_beta015.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)

In [ ]:
import pickle
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

cell_summary = pd.read_csv(REPORT_DIR / 'cell_level_anomaly_summary.csv')

def extract_batch_id(cell_id):
    m = re.search(r'(b\d+)', str(cell_id))
    return m.group(1) if m else 'unknown'

def extract_cell_id_from_path(path):
    return Path(path).stem

def numeric_candidates(obj, prefix=''):
    out = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f'{prefix}.{k}' if prefix else str(k)
            out.extend(numeric_candidates(v, key))

    elif isinstance(obj, (list, tuple, np.ndarray)):
        arr = np.asarray(obj)

        if arr.dtype == object:
            return out

        if np.issubdtype(arr.dtype, np.number):
            arr = arr.astype(float)
            if arr.ndim == 0:
                out.append((prefix, float(arr)))
            elif arr.ndim == 1 and len(arr) > 0:
                finite = arr[np.isfinite(arr)]
                if len(finite) > 0:
                    out.append((prefix, finite))

    elif isinstance(obj, (int, float, np.integer, np.floating)):
        out.append((prefix, float(obj)))

    return out

def pick_capacity_from_cycle(cycle_obj):
    candidates = numeric_candidates(cycle_obj)

    # capacity 계열 키 우선. voltage/current/time 같은 건 제외.
    keywords = [
        'qdischarge',
        'discharge_capacity',
        'capacity',
        'qd',
        'q_d',
    ]

    exclude = [
        'nominal',
        'voltage',
        'current',
        'temperature',
        'time',
        'soc',
        'dod',
        'doc',
    ]

    scored = []
    for name, value in candidates:
        lname = name.lower()

        if any(e in lname for e in exclude):
            continue

        score = sum(1 for kw in keywords if kw in lname)
        if score <= 0:
            continue

        arr = np.asarray(value, dtype=float)
        finite = arr[np.isfinite(arr)]

        if len(finite) == 0:
            continue

        # cycle 내 capacity trace면 마지막 양수값 또는 최대값 사용
        positive = finite[finite > 0]
        if len(positive) == 0:
            continue

        cap_value = float(positive[-1])
        if cap_value <= 0:
            cap_value = float(np.nanmax(positive))

        scored.append((score, name, cap_value))

    if not scored:
        return np.nan, None

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][2], scored[0][1]

def load_matr_soh_curve(pkl_path, debug=False):
    with open(pkl_path, 'rb') as f:
        obj = pickle.load(f)

    if not isinstance(obj, dict):
        raise ValueError(f'Unsupported pkl type: {type(obj)}')

    if 'cycle_data' not in obj:
        raise ValueError(f'No cycle_data key. keys={list(obj.keys())[:20]}')

    cycle_data = obj['cycle_data']

    if debug:
        print('file:', pkl_path)
        print('top-level keys:', list(obj.keys())[:30])
        print('cycle_data type:', type(cycle_data), 'len:', len(cycle_data))
        first = cycle_data[0]
        print('first cycle type:', type(first))
        if isinstance(first, dict):
            print('first cycle keys:', list(first.keys())[:50])
            print('numeric candidates in first cycle:')
            for name, value in numeric_candidates(first)[:80]:
                arr = np.asarray(value)
                print(' ', name, arr.shape if arr.ndim else 'scalar')

    cycles = []
    capacities = []
    used_keys = []

    for i, cyc in enumerate(cycle_data):
        if not isinstance(cyc, dict):
            continue

        cap, used_key = pick_capacity_from_cycle(cyc)
        if not np.isfinite(cap):
            continue

        # cycle 번호가 cycle dict 안에 있으면 사용, 아니면 index 사용
        cycle_num = None
        for key in ['cycle', 'cycle_index', 'cycle_number', 'Cycle_Index']:
            if key in cyc:
                try:
                    cycle_num = float(cyc[key])
                    break
                except Exception:
                    pass

        if cycle_num is None:
            cycle_num = i + 1 + float(obj.get('already_spent_cycles', 0) or 0)

        cycles.append(cycle_num)
        capacities.append(cap)
        used_keys.append(used_key)

    if len(capacities) == 0:
        raise ValueError('No capacity values extracted from cycle_data')

    cycles = np.asarray(cycles, dtype=float)
    capacities = np.asarray(capacities, dtype=float)

    valid = np.isfinite(cycles) & np.isfinite(capacities) & (capacities > 0)
    cycles = cycles[valid]
    capacities = capacities[valid]

    initial_capacity = capacities[0]
    soh = capacities / initial_capacity

    curve = pd.DataFrame({
        'cycle': cycles,
        'soh': soh,
        'capacity': capacities,
    }).sort_values('cycle')

    if debug:
        print('extracted points:', len(curve))
        print('used capacity key examples:', pd.Series(used_keys).value_counts().head())

    return curve

# =========================
# 1. 후보 cell 선택
# =========================
cell_summary['batch_id'] = cell_summary['cell_id'].apply(extract_batch_id)

candidate_cells = cell_summary[cell_summary['is_cell_anomaly'] == True].copy()

if candidate_cells.empty:
    candidate_cells = cell_summary.sort_values(
        ['anomaly_ratio', 'max_anomaly_score'],
        ascending=[False, False]
    ).head(2).copy()
else:
    candidate_cells = candidate_cells.sort_values(
        ['anomaly_ratio', 'max_anomaly_score'],
        ascending=[False, False]
    ).head(2).copy()

target_batch = candidate_cells['batch_id'].mode().iloc[0]
candidate_ids = set(candidate_cells['cell_id'].astype(str))

print('target_batch:', target_batch)
print('candidate_ids:', candidate_ids)
display(candidate_cells)

# =========================
# 2. 같은 batch raw pkl curve 로드
# =========================
pkl_paths = sorted(MATR_DIR.rglob('*.pkl'))

sample = next(
    (p for p in pkl_paths if extract_batch_id(extract_cell_id_from_path(p)) == target_batch),
    None
)

if sample is not None:
    _ = load_matr_soh_curve(sample, debug=True)

records = []
failed = []

for p in pkl_paths:
    cell_id = extract_cell_id_from_path(p)
    batch_id = extract_batch_id(cell_id)

    if batch_id != target_batch:
        continue

    try:
        curve = load_matr_soh_curve(p)
        curve['cell_id'] = cell_id
        curve['batch_id'] = batch_id
        records.append(curve)
    except Exception as e:
        failed.append((str(p), str(e)))

print('loaded curves:', len(records))
print('failed curves:', len(failed))

if failed[:5]:
    print('failed examples:')
    for item in failed[:5]:
        print(item)

if not records:
    raise RuntimeError('No raw SOH curves loaded.')

raw_curves = pd.concat(records, ignore_index=True)

print('same batch cells:', raw_curves['cell_id'].nunique())
print('cycle range:', raw_curves['cycle'].min(), raw_curves['cycle'].max())

# =========================
# 3. Plot: x축 2000 cycle
# =========================
plt.figure(figsize=(11, 6))

for cell_id, one in raw_curves.groupby('cell_id'):
    one = one.sort_values('cycle')

    if cell_id in candidate_ids:
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color='lightgray',
        linewidth=1.0,
        alpha=0.55
    )

candidate_colors = ['#2563EB', '#F97316', '#DC2626', '#16A34A']

for idx, cell_id in enumerate(candidate_ids):
    one = raw_curves[raw_curves['cell_id'] == cell_id].sort_values('cycle')

    if one.empty:
        print('candidate raw curve not found:', cell_id)
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color=candidate_colors[idx % len(candidate_colors)],
        linewidth=2.8,
        label=f'{cell_id} anomaly candidate'
    )

plt.xlabel('Cycle')
plt.ylabel('SOH')
plt.title(f'SOH Curves: {target_batch} anomaly candidates vs normal-like cells')
plt.xlim(0, 1000)
plt.ylim(0.80, 1.01)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

save_path = FIGURE_DIR / f'batch_{target_batch}_candidate_vs_normal_full_1000cycles.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)

In [ ]:
# =========================
# b3 batch candidate vs normal-like cells
# =========================

target_batch = 'b3'
xmax = 2000

# b3 안의 anomaly candidate 선택
b3_candidates = cell_summary[
    (cell_summary['batch_id'] == target_batch) &
    (cell_summary['is_cell_anomaly'] == True)
].copy()

# 만약 b3에서 threshold 기준 candidate가 없으면 score 상위 2개를 후보처럼 표시
if b3_candidates.empty:
    b3_candidates = (
        cell_summary[cell_summary['batch_id'] == target_batch]
        .sort_values(['anomaly_ratio', 'max_anomaly_score'], ascending=[False, False])
        .head(2)
        .copy()
    )
else:
    b3_candidates = (
        b3_candidates
        .sort_values(['anomaly_ratio', 'max_anomaly_score'], ascending=[False, False])
        .head(2)
        .copy()
    )

b3_candidate_ids = set(b3_candidates['cell_id'].astype(str))

print('target_batch:', target_batch)
print('b3_candidate_ids:', b3_candidate_ids)
display(b3_candidates)

# b3 raw curves 로드
pkl_paths = sorted(MATR_DIR.rglob('*.pkl'))

records = []
failed = []

for p in pkl_paths:
    cell_id = extract_cell_id_from_path(p)
    batch_id = extract_batch_id(cell_id)

    if batch_id != target_batch:
        continue

    try:
        curve = load_matr_soh_curve(p)
        curve['cell_id'] = cell_id
        curve['batch_id'] = batch_id
        records.append(curve)
    except Exception as e:
        failed.append((str(p), str(e)))

print('loaded b3 curves:', len(records))
print('failed b3 curves:', len(failed))

if failed[:5]:
    print('failed examples:')
    for item in failed[:5]:
        print(item)

if not records:
    raise RuntimeError('No b3 raw SOH curves loaded.')

b3_raw_curves = pd.concat(records, ignore_index=True)

print('b3 cells:', b3_raw_curves['cell_id'].nunique())
print('b3 cycle range:', b3_raw_curves['cycle'].min(), b3_raw_curves['cycle'].max())

# plot
plt.figure(figsize=(11, 6))

# normal-like cells: gray
for cell_id, one in b3_raw_curves.groupby('cell_id'):
    one = one.sort_values('cycle')

    if cell_id in b3_candidate_ids:
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color='lightgray',
        linewidth=1.0,
        alpha=0.55
    )

# candidates: colored
candidate_colors = ['#2563EB', '#F97316', '#DC2626', '#16A34A']

for idx, cell_id in enumerate(b3_candidate_ids):
    one = b3_raw_curves[b3_raw_curves['cell_id'] == cell_id].sort_values('cycle')

    if one.empty:
        print('candidate raw curve not found:', cell_id)
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color=candidate_colors[idx % len(candidate_colors)],
        linewidth=2.8,
        label=f'{cell_id} anomaly candidate'
    )

plt.xlabel('Cycle')
plt.ylabel('SOH')
plt.title(f'SOH Curves: {target_batch} anomaly candidates vs normal-like cells')
plt.xlim(0, xmax)
plt.ylim(0.80, 1.01)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

save_path = FIGURE_DIR / f'batch_{target_batch}_candidate_vs_normal_full_{xmax}cycles.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)